# Reproducing the Analysis from Ng, Jordan & Weiss (2001)

**"On Spectral Clustering: Analysis and an Algorithm"**

This notebook reproduces the key experiments and figures from the NJW paper, then extends them into the CASA (Computational Auditory Scene Analysis) domain — showing spectral clustering as a principled mechanism for auditory stream segregation.

### Figures

1. **Concentric circles** — k-means fails, spectral clustering succeeds
2. **Interleaved half-moons** — same story
3. **Eigenvector embedding** — what the data looks like in Laplacian eigenspace
4. **Eigenvalue spectrum** — the eigengap heuristic for choosing k
5. **CASA application** — spectral clustering on a cochlear filterbank similarity graph, recovering auditory streams from a mixture

*Erick Oduniyi — Chaos Hearing / CASA*

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import sparse
from scipy.spatial.distance import pdist, squareform
from sklearn.cluster import KMeans
from sklearn.datasets import make_circles, make_moons

from casa.graphs import spectral_clustering, graph_laplacian, spectral_analysis
from casa.oscillators import cochlear_filterbank

%matplotlib inline
plt.rcParams['figure.dpi'] = 130
plt.rcParams['font.size'] = 10

In [ ]:
# ── Colour palette & helpers ─────────────────────────────────────
C0 = '#E91E63'   # magenta-pink
C1 = '#2196F3'   # blue
C2 = '#4CAF50'   # green
C3 = '#FF9800'   # orange
C4 = '#9C27B0'   # purple
GREY = '#9E9E9E'
COLORS = [C0, C1, C2, C3, C4]

def gaussian_affinity(X, sigma):
    """Build a Gaussian (RBF) affinity matrix from point cloud X."""
    D = squareform(pdist(X, 'sqeuclidean'))
    W = np.exp(-D / (2.0 * sigma ** 2))
    np.fill_diagonal(W, 0.0)
    return sparse.csr_matrix(W)

---
## Figure 1 — Concentric Circles

The motivating example from NJW §4. Two concentric rings are not linearly separable — k-means in $\mathbb{R}^2$ draws a straight cut through both rings. Spectral clustering embeds the points via the graph Laplacian eigenvectors, where the rings become separable, then runs k-means in that space.

In [ ]:
np.random.seed(42)
X_circles, y_circles = make_circles(n_samples=300, factor=0.4, noise=0.06)

# k-means in original space
labels_km = KMeans(n_clusters=2, random_state=42, n_init=10).fit_predict(X_circles)

# NJW spectral clustering
W_circles = gaussian_affinity(X_circles, sigma=0.2)
labels_sc, emb_circles = spectral_clustering(W_circles, n_clusters=2)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
fig.suptitle('Figure 1 — Concentric Circles (NJW §4)', fontsize=13, y=1.02)

for ax, labels, title in [
    (axes[0], y_circles,  '(a) Ground truth'),
    (axes[1], labels_km,  '(b) k-means in ℝ²'),
    (axes[2], labels_sc,  '(c) NJW spectral clustering'),
]:
    for c in [0, 1]:
        mask = labels == c
        ax.scatter(X_circles[mask, 0], X_circles[mask, 1],
                   s=12, c=COLORS[c], alpha=0.7, edgecolors='none')
    ax.set_title(title)
    ax.set_aspect('equal')
    ax.set_xlabel('x₁'); ax.set_ylabel('x₂')

plt.tight_layout()
plt.show()

---
## Figure 2 — Interleaved Half-Moons

Another classic non-convex geometry. The two half-moons interlock — k-means can only find a linear boundary, while spectral clustering respects the manifold structure.

In [ ]:
np.random.seed(7)
X_moons, y_moons = make_moons(n_samples=300, noise=0.07)

labels_km_m = KMeans(n_clusters=2, random_state=42, n_init=10).fit_predict(X_moons)

W_moons = gaussian_affinity(X_moons, sigma=0.25)
labels_sc_m, _ = spectral_clustering(W_moons, n_clusters=2)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
fig.suptitle('Figure 2 — Interleaved Half-Moons (NJW §4)', fontsize=13, y=1.02)

for ax, labels, title in [
    (axes[0], y_moons,     '(a) Ground truth'),
    (axes[1], labels_km_m, '(b) k-means in ℝ²'),
    (axes[2], labels_sc_m, '(c) NJW spectral clustering'),
]:
    for c in [0, 1]:
        mask = labels == c
        ax.scatter(X_moons[mask, 0], X_moons[mask, 1],
                   s=12, c=COLORS[c], alpha=0.7, edgecolors='none')
    ax.set_title(title)
    ax.set_aspect('equal')
    ax.set_xlabel('x₁'); ax.set_ylabel('x₂')

plt.tight_layout()
plt.show()

---
## Figure 3 — Eigenvector Embedding

The key insight from NJW §3: after projecting onto the top-$k$ eigenvectors of $D^{-1/2} W D^{-1/2}$ and row-normalising, the clusters that were interleaved in $\mathbb{R}^2$ become linearly separable in the eigenvector space. This is *why* k-means works in the embedding — the Laplacian eigenvectors unfold the manifold.

In [ ]:
labels_sc_e, embedding = spectral_clustering(W_circles, n_clusters=2)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
fig.suptitle('Figure 3 — Eigenvector Embedding (NJW §3)', fontsize=13, y=1.02)

# (a) Original space
for c in [0, 1]:
    mask = labels_sc_e == c
    axes[0].scatter(X_circles[mask, 0], X_circles[mask, 1],
                    s=14, c=COLORS[c], alpha=0.7, edgecolors='none')
axes[0].set_title('(a) Original space ℝ²')
axes[0].set_xlabel('x₁'); axes[0].set_ylabel('x₂')
axes[0].set_aspect('equal')

# (b) Eigenvector space
for c in [0, 1]:
    mask = labels_sc_e == c
    axes[1].scatter(embedding[mask, 0], embedding[mask, 1],
                    s=14, c=COLORS[c], alpha=0.7, edgecolors='none')
axes[1].set_title('(b) Eigenvector space (row-normalised)')
axes[1].set_xlabel('v₁ (1st eigenvector)')
axes[1].set_ylabel('v₂ (2nd eigenvector)')
axes[1].set_aspect('equal')
axes[1].annotate('linearly\nseparable', xy=(0.5, 0.5),
                 xycoords='axes fraction', fontsize=9,
                 color=GREY, ha='center', fontstyle='italic')

plt.tight_layout()
plt.show()

---
## Figure 4 — Eigenvalue Spectrum & Eigengap Heuristic

The number of clusters $k$ can be read off the Laplacian eigenvalue spectrum: look for the largest gap between consecutive eigenvalues. For a graph with $k$ well-separated clusters, the first $k$ eigenvalues are near zero (one per connected component), then there's a jump.

Here we use 3 Gaussian blobs to show the eigengap at $k = 3$.

In [ ]:
np.random.seed(12)
n_per = 80
blob_a = np.random.randn(n_per, 2) * 0.3 + np.array([-2, 0])
blob_b = np.random.randn(n_per, 2) * 0.3 + np.array([2, 0])
blob_c = np.random.randn(n_per, 2) * 0.3 + np.array([0, 2.5])
X_blobs = np.vstack([blob_a, blob_b, blob_c])

W_blobs = gaussian_affinity(X_blobs, sigma=0.5)
L_blobs, _ = graph_laplacian(W_blobs)
eigenvalues, _ = spectral_analysis(L_blobs, k=15)
labels_blobs, _ = spectral_clustering(W_blobs, n_clusters=3)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
fig.suptitle('Figure 4 — Eigenvalue Spectrum & Eigengap Heuristic',
             fontsize=13, y=1.02)

# (a) Clustering result
for c in range(3):
    mask = labels_blobs == c
    axes[0].scatter(X_blobs[mask, 0], X_blobs[mask, 1],
                    s=14, c=COLORS[c], alpha=0.7, edgecolors='none')
axes[0].set_title('(a) 3 clusters — NJW result')
axes[0].set_aspect('equal')
axes[0].set_xlabel('x₁'); axes[0].set_ylabel('x₂')

# (b) Eigenvalue spectrum
k_plot = min(12, len(eigenvalues))
axes[1].bar(range(k_plot), eigenvalues[:k_plot],
            color=GREY, alpha=0.7, edgecolor='black', linewidth=0.5)
if k_plot > 3:
    gap_val = eigenvalues[3] - eigenvalues[2]
    axes[1].annotate(f'eigengap\nΔλ = {gap_val:.3f}',
                     xy=(3, eigenvalues[3]),
                     xytext=(5, eigenvalues[3] + 0.02),
                     fontsize=9, color=C0,
                     arrowprops=dict(arrowstyle='->', color=C0, lw=1.2))
    for i in range(3):
        axes[1].patches[i].set_facecolor(C1)
        axes[1].patches[i].set_alpha(0.85)
axes[1].set_xlabel('Eigenvalue index i')
axes[1].set_ylabel('λᵢ')
axes[1].set_title('(b) Laplacian eigenvalues')

# (c) Eigengap plot
diffs = np.diff(eigenvalues[:k_plot])
axes[2].bar(range(1, len(diffs)+1), diffs,
            color=GREY, alpha=0.7, edgecolor='black', linewidth=0.5)
if len(diffs) > 2:
    max_gap_idx = np.argmax(diffs)
    axes[2].patches[max_gap_idx].set_facecolor(C0)
    axes[2].patches[max_gap_idx].set_alpha(0.9)
    axes[2].annotate(f'k = {max_gap_idx + 1}',
                     xy=(max_gap_idx + 1, diffs[max_gap_idx]),
                     xytext=(max_gap_idx + 2.5, diffs[max_gap_idx]),
                     fontsize=10, color=C0, fontweight='bold',
                     arrowprops=dict(arrowstyle='->', color=C0, lw=1.2))
axes[2].set_xlabel('Gap index (λᵢ₊₁ − λᵢ)')
axes[2].set_ylabel('Δλ')
axes[2].set_title('(c) Eigengap heuristic → k = 3')

plt.tight_layout()
plt.show()

---
## Figure 5 — CASA: Spectral Clustering as Auditory Stream Segregation

Now we connect NJW to auditory scene analysis. The setup:

1. **Two harmonic sources** — Source A (f₀ = 220 Hz, harmonics at 440, 660, 880 Hz) and Source B (f₀ = 330 Hz, harmonics at 660, 990 Hz) are mixed together.
2. **Cochlear filterbank** — The mixture is passed through a bank of 48 Hopf oscillators (the `casa.oscillators` module), producing amplitude envelopes at each characteristic frequency.
3. **Affinity matrix** — We compute pairwise correlation between channel envelopes. Channels driven by the same source will have correlated responses (this is the *common modulation* grouping cue from Bregman's Auditory Scene Analysis).
4. **NJW spectral clustering** — Partition the channels into 2 streams.

The result: spectral clustering recovers the two auditory streams from the mixture — the same algorithm that separates concentric circles now separates harmonic sources. This is the cocktail party problem, solved by graph partitioning.

In [ ]:
np.random.seed(0)

# Two harmonic sources
sr = 16000
duration = 0.1
t = np.linspace(0, duration, int(sr * duration), endpoint=False)

# Source A: fundamental 220 Hz + harmonics (2f, 3f, 4f)
source_a = (np.sin(2 * np.pi * 220 * t)
            + 0.6 * np.sin(2 * np.pi * 440 * t)
            + 0.3 * np.sin(2 * np.pi * 660 * t)
            + 0.15 * np.sin(2 * np.pi * 880 * t))

# Source B: fundamental 330 Hz + harmonics
source_b = (0.8 * np.sin(2 * np.pi * 330 * t)
            + 0.5 * np.sin(2 * np.pi * 660 * t)
            + 0.25 * np.sin(2 * np.pi * 990 * t))

mixture = source_a + source_b + 0.02 * np.random.randn(len(t))

# Cochlear filterbank (48 Hopf oscillators)
n_channels = 48
responses, freqs = cochlear_filterbank(
    mixture, sr, n_channels=n_channels,
    f_low=100, f_high=2000, mu=-0.05
)

# Affinity matrix: correlation between channel envelopes
# (common modulation = Bregman grouping cue)
norms = np.linalg.norm(responses, axis=1, keepdims=True)
norms = np.where(norms > 0, norms, 1.0)
envelope_norm = responses / norms
corr = np.maximum(envelope_norm @ envelope_norm.T, 0)
np.fill_diagonal(corr, 0)
W_casa = sparse.csr_matrix(corr)

# NJW spectral clustering -> 2 streams
labels_casa, emb_casa = spectral_clustering(W_casa, n_clusters=2)

# ── 9-panel figure ──────────────────────────────────────────────
fig = plt.figure(figsize=(15, 10))
fig.suptitle("Figure 5 — CASA: Spectral Clustering as Auditory Stream Segregation",
             fontsize=13, y=0.98)
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.4)
extent = [0, duration * 1000, freqs[0], freqs[-1]]

# (a) Mixture waveform
ax = fig.add_subplot(gs[0, 0])
ax.plot(t * 1000, mixture, linewidth=0.4, color="steelblue")
ax.set_title("(a) Acoustic mixture")
ax.set_xlabel("Time (ms)"); ax.set_ylabel("Amplitude")

# (b) Individual sources
ax = fig.add_subplot(gs[0, 1])
ax.plot(t * 1000, source_a, linewidth=0.5, alpha=0.7, color=C0, label="Source A (f₀=220 Hz)")
ax.plot(t * 1000, source_b, linewidth=0.5, alpha=0.7, color=C1, label="Source B (f₀=330 Hz)")
ax.set_title("(b) Ground truth sources")
ax.set_xlabel("Time (ms)"); ax.set_ylabel("Amplitude")
ax.legend(fontsize=7)

# (c) Cochleagram
ax = fig.add_subplot(gs[0, 2])
ax.imshow(responses, aspect="auto", origin="lower", cmap="inferno",
          extent=extent, interpolation="nearest")
ax.set_title("(c) Cochlear filterbank response")
ax.set_xlabel("Time (ms)"); ax.set_ylabel("Frequency (Hz)")

# (d) Affinity matrix
ax = fig.add_subplot(gs[1, 0])
im = ax.imshow(corr, cmap="YlOrRd", origin="lower", aspect="auto")
ax.set_title("(d) Channel correlation (affinity W)")
ax.set_xlabel("Channel"); ax.set_ylabel("Channel")
plt.colorbar(im, ax=ax, fraction=0.046)

# (e) Eigenvector embedding
ax = fig.add_subplot(gs[1, 1])
for c in range(2):
    mask = labels_casa == c
    ax.scatter(emb_casa[mask, 0], emb_casa[mask, 1], s=30,
               c=COLORS[c], alpha=0.8, edgecolors="black", linewidth=0.3,
               label=f"Stream {c}")
ax.set_title("(e) Eigenvector embedding")
ax.set_xlabel("v₁"); ax.set_ylabel("v₂")
ax.legend(fontsize=8)

# (f) Eigenvalue spectrum
ax = fig.add_subplot(gs[1, 2])
L_casa, _ = graph_laplacian(W_casa)
evals_casa, _ = spectral_analysis(L_casa, k=min(15, n_channels - 2))
k_plot = min(12, len(evals_casa))
bars = ax.bar(range(k_plot), evals_casa[:k_plot], color=GREY, alpha=0.7,
              edgecolor="black", linewidth=0.5)
if k_plot > 2:
    bars[0].set_facecolor(C1); bars[1].set_facecolor(C1)
    bars[0].set_alpha(0.85); bars[1].set_alpha(0.85)
    gap = evals_casa[2] - evals_casa[1]
    ax.annotate(f"eigengap\nΔλ = {gap:.2f}", xy=(2, evals_casa[2]),
                xytext=(4, evals_casa[2] + 0.5), fontsize=8, color=C0,
                arrowprops=dict(arrowstyle="->", color=C0, lw=1))
ax.set_xlabel("Eigenvalue index"); ax.set_ylabel("λᵢ")
ax.set_title("(f) Laplacian eigenvalues")

# (g) Stream assignments by frequency
ax = fig.add_subplot(gs[2, 0])
for c in range(2):
    mask = labels_casa == c
    ax.barh(np.where(mask)[0], freqs[mask], color=COLORS[c], alpha=0.8, height=0.8, label=f"Stream {c}")
ax.set_xlabel("Characteristic frequency (Hz)"); ax.set_ylabel("Channel index")
ax.set_title("(g) Stream assignments by frequency"); ax.legend(fontsize=8)

# (h) Separated cochleagrams
ax = fig.add_subplot(gs[2, 1])
stream_0 = responses.copy(); stream_0[labels_casa != 0] = 0
ax.imshow(stream_0, aspect="auto", origin="lower", cmap="Reds",
          extent=extent, interpolation="nearest", alpha=0.9)
ax.set_title(f"(h) Stream 0 — {np.sum(labels_casa==0)} channels")
ax.set_xlabel("Time (ms)"); ax.set_ylabel("Frequency (Hz)")

ax = fig.add_subplot(gs[2, 2])
stream_1 = responses.copy(); stream_1[labels_casa != 1] = 0
ax.imshow(stream_1, aspect="auto", origin="lower", cmap="Blues",
          extent=extent, interpolation="nearest", alpha=0.9)
ax.set_title(f"(i) Stream 1 — {np.sum(labels_casa==1)} channels")
ax.set_xlabel("Time (ms)"); ax.set_ylabel("Frequency (Hz)")

plt.show()